In [ ]:

import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf

from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

AUTOTUNE = tf.data.AUTOTUNE

IMG_SIZE = (224, 224)
NUM_CLASSES = 37

EPOCHS = 5

print("Seed:", SEED)
print("Image size:", IMG_SIZE)
print("Number of classes:", NUM_CLASSES)

In [ ]:

import tensorflow_datasets as tfds

(ds_train_full, ds_test), ds_info = tfds.load(
    "oxford_iiit_pet",
    split=["train", "test"],
    as_supervised=True,
    with_info=True
)

print("Training samples:", ds_info.splits["train"].num_examples)
print("Test samples:", ds_info.splits["test"].num_examples)
print("Number of classes:", ds_info.features["label"].num_classes)

In [ ]:

TRAIN_SIZE = 0.8

total_train = ds_info.splits["train"].num_examples
train_count = int(total_train * TRAIN_SIZE)

ds_train_full = ds_train_full.shuffle(
    total_train,
    seed=SEED,
    reshuffle_each_iteration=False
)

ds_train = ds_train_full.take(train_count)
ds_val = ds_train_full.skip(train_count)

print("Train:", train_count)
print("Validation:", total_train - train_count)
print("Test:", ds_info.splits["test"].num_examples)

In [ ]:

def preprocess_image(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)

    image = preprocess_input(image)

    return image, label


def prepare_dataset(dataset, batch_size=32, shuffle=False):
    dataset = dataset.map(
        preprocess_image,
        num_parallel_calls=AUTOTUNE
    )

    if shuffle:
        dataset = dataset.shuffle(
            1000,
            seed=SEED
        )

    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(AUTOTUNE)

    return dataset

In [ ]:

plt.figure(figsize=(12, 8))

for i, (image, label) in enumerate(ds_train.take(12)):
    plt.subplot(3, 4, i + 1)

    plt.imshow(image)
    plt.title(ds_info.features["label"].int2str(label.numpy()))
    plt.axis("off")

plt.suptitle("Oxford-IIIT Pet Dataset Samples")
plt.tight_layout()
plt.show()

In [ ]:

def build_mobilenet_model(
    initialization="he",
    dropout_rate=0.0,
    l2_strength=0.0,
    use_bn=True,
    optimizer_name="adam",
    learning_rate=1e-3,
    train_base=False,
    fine_tune_layers=0
):

    # --------------------------------------------------------
    # MobileNetV2 base
    # --------------------------------------------------------

    base = MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights="imagenet"
    )

    base.trainable = train_base

    # Fine-tuning
    if train_base and fine_tune_layers > 0:

        for layer in base.layers[:-fine_tune_layers]:
            layer.trainable = False

        for layer in base.layers[-fine_tune_layers:]:
            layer.trainable = True

    # --------------------------------------------------------
    # Classifier
    # --------------------------------------------------------

    inputs = layers.Input(shape=(224, 224, 3))

    x = base(inputs, training=train_base)

    x = layers.GlobalAveragePooling2D()(x)

    # Batch Normalization
    if use_bn:
        x = layers.BatchNormalization()(x)

    # Dropout
    if dropout_rate > 0:
        x = layers.Dropout(dropout_rate)(x)

    # Regularization
    kernel_regularizer = None

    if l2_strength > 0:
        kernel_regularizer = regularizers.l2(l2_strength)

    # Initialization
    if initialization == "zero":
        initializer = tf.keras.initializers.Zeros()

    elif initialization == "random":
        initializer = tf.keras.initializers.RandomNormal(
            mean=0.0,
            stddev=0.05,
            seed=SEED
        )

    elif initialization == "xavier":
        initializer = tf.keras.initializers.GlorotUniform(
            seed=SEED
        )

    elif initialization == "he":
        initializer = tf.keras.initializers.HeNormal(
            seed=SEED
        )

    else:
        raise ValueError("Unknown initialization")

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        kernel_initializer=initializer,
        kernel_regularizer=kernel_regularizer
    )(x)

    model = models.Model(inputs, outputs)

    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    if optimizer_name == "sgd":

        optimizer = tf.keras.optimizers.SGD(
            learning_rate=learning_rate
        )

    elif optimizer_name == "momentum":

        optimizer = tf.keras.optimizers.SGD(
            learning_rate=learning_rate,
            momentum=0.9
        )

    elif optimizer_name == "rmsprop":

        optimizer = tf.keras.optimizers.RMSprop(
            learning_rate=learning_rate
        )

    elif optimizer_name == "adam":

        optimizer = tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        )

    else:
        raise ValueError("Unknown optimizer")

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:

def train_model(
    model,
    train_dataset,
    val_dataset,
    epochs=EPOCHS,
    verbose=1
):

    start_time = time.time()

    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=epochs,
        verbose=verbose
    )

    training_time = time.time() - start_time

    return history, training_time

In [ ]:

def plot_loss_comparison(histories, title="Training Loss"):
    plt.figure(figsize=(10, 6))

    for name, history in histories.items():
        plt.plot(
            history.history["loss"],
            label=name
        )

    plt.xlabel("Epoch")
    plt.ylabel("Training Loss")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()


def plot_val_accuracy_comparison(
    histories,
    title="Validation Accuracy"
):

    plt.figure(figsize=(10, 6))

    for name, history in histories.items():
        plt.plot(
            np.array(history.history["val_accuracy"]) * 100,
            label=name
        )

    plt.xlabel("Epoch")
    plt.ylabel("Validation Accuracy (%)")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()


def plot_train_val_accuracy(history, title):

    plt.figure(figsize=(10, 6))

    plt.plot(
        np.array(history.history["accuracy"]) * 100,
        label="Training Accuracy"
    )

    plt.plot(
        np.array(history.history["val_accuracy"]) * 100,
        label="Validation Accuracy"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()


def plot_train_val_loss(history, title):

    plt.figure(figsize=(10, 6))

    plt.plot(
        history.history["loss"],
        label="Training Loss"
    )

    plt.plot(
        history.history["val_loss"],
        label="Validation Loss"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:

train_ds = prepare_dataset(
    ds_train,
    batch_size=32,
    shuffle=True
)

val_ds = prepare_dataset(
    ds_val,
    batch_size=32,
    shuffle=False
)

test_ds = prepare_dataset(
    ds_test,
    batch_size=32,
    shuffle=False
)

baseline_model = build_mobilenet_model(
    initialization="he",
    dropout_rate=0,
    l2_strength=0,
    use_bn=True,
    optimizer_name="adam",
    learning_rate=1e-3,
    train_base=False
)

baseline_model.summary()

In [ ]:

baseline_history, baseline_time = train_model(
    baseline_model,
    train_ds,
    val_ds
)

print("Training time:", baseline_time, "seconds")

In [ ]:
plot_train_val_accuracy(
    baseline_history,
    "Baseline - Training vs Validation Accuracy"
)

plot_train_val_loss(
    baseline_history,
    "Baseline - Training vs Validation Loss"
)

In [ ]:

initializations = [
    "zero",
    "random",
    "xavier",
    "he"
]

init_histories = {}
init_results = {}

for init in initializations:

    print("Initialization:", init)

    model = build_mobilenet_model(
        initialization=init,
        dropout_rate=0,
        l2_strength=0,
        use_bn=True,
        optimizer_name="adam",
        learning_rate=1e-3,
        train_base=False
    )

    history, training_time = train_model(
        model,
        train_ds,
        val_ds,
        epochs=EPOCHS
    )

    init_histories[init] = history

    init_results[init] = {
        "Best Validation Accuracy":
            max(history.history["val_accuracy"]) * 100,

        "Final Training Loss":
            history.history["loss"][-1],

        "Training Time":
            training_time
    }

init_results_df = pd.DataFrame(init_results).T

display(init_results_df)

In [ ]:
plot_loss_comparison(
    init_histories,
    "Plot 1 - Training Loss for Different Initializations"
)

In [ ]:

plot_val_accuracy_comparison(
    init_histories,
    "Plot 2 - Validation Accuracy for Different Initializations"
)

In [ ]:

regularization_configs = {

    "No Regularization": {
        "l2_strength": 0,
        "dropout_rate": 0,
        "use_bn": False
    },

    "L2": {
        "l2_strength": 1e-4,
        "dropout_rate": 0,
        "use_bn": False
    },

    "Dropout": {
        "l2_strength": 0,
        "dropout_rate": 0.5,
        "use_bn": False
    },

    "Batch Normalization": {
        "l2_strength": 0,
        "dropout_rate": 0,
        "use_bn": True
    }
}

regularization_histories = {}
regularization_results = {}

for name, config in regularization_configs.items():

    print(name)
    model = build_mobilenet_model(
        initialization="he",
        optimizer_name="adam",
        learning_rate=1e-3,
        train_base=False,
        **config
    )

    history, training_time = train_model(
        model,
        train_ds,
        val_ds
    )

    regularization_histories[name] = history

    regularization_results[name] = {
        "Best Val Accuracy":
            max(history.history["val_accuracy"]) * 100,

        "Final Train Accuracy":
            history.history["accuracy"][-1] * 100,

        "Final Val Accuracy":
            history.history["val_accuracy"][-1] * 100,

        "Final Train Loss":
            history.history["loss"][-1],

        "Final Val Loss":
            history.history["val_loss"][-1],

        "Training Time":
            training_time
    }

regularization_df = pd.DataFrame(
    regularization_results
).T

display(regularization_df)

In [ ]:

plt.figure(figsize=(12, 7))

for name, history in regularization_histories.items():

    plt.plot(
        np.array(history.history["accuracy"]) * 100,
        label=f"{name} - Train"
    )

    plt.plot(
        np.array(history.history["val_accuracy"]) * 100,
        linestyle="--",
        label=f"{name} - Validation"
    )

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Plot 3 - Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:

plt.figure(figsize=(12, 7))

for name, history in regularization_histories.items():

    plt.plot(
        history.history["loss"],
        label=f"{name} - Train"
    )

    plt.plot(
        history.history["val_loss"],
        linestyle="--",
        label=f"{name} - Validation"
    )

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Plot 4 - Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:

bn_configs = {
    "With BN": True,
    "Without BN": False
}

bn_histories = {}

for name, use_bn in bn_configs.items():

    model = build_mobilenet_model(
        initialization="he",
        dropout_rate=0,
        l2_strength=0,
        use_bn=use_bn,
        optimizer_name="adam",
        learning_rate=1e-3,
        train_base=False
    )

    history, _ = train_model(
        model,
        train_ds,
        val_ds
    )

    bn_histories[name] = history

In [ ]:

plot_val_accuracy_comparison(
    bn_histories,
    "Plot 5 - With vs Without Batch Normalization"
)

In [ ]:

optimizers = [
    "sgd",
    "momentum",
    "rmsprop",
    "adam"
]

optimizer_histories = {}
optimizer_results = {}

for optimizer_name in optimizers:
    print("Optimizer:", optimizer_name)

    model = build_mobilenet_model(
        initialization="he",
        dropout_rate=0,
        l2_strength=0,
        use_bn=True,
        optimizer_name=optimizer_name,
        learning_rate=1e-3,
        train_base=False
    )

    history, training_time = train_model(
        model,
        train_ds,
        val_ds
    )

    optimizer_histories[optimizer_name] = history

    optimizer_results[optimizer_name] = {
        "Final Loss":
            history.history["loss"][-1],

        "Best Val Accuracy":
            max(history.history["val_accuracy"]) * 100,

        "Final Val Accuracy":
            history.history["val_accuracy"][-1] * 100,

        "Training Time":
            training_time
    }

optimizer_df = pd.DataFrame(
    optimizer_results
).T

display(optimizer_df)

In [ ]:

plot_loss_comparison(
    optimizer_histories,
    "Plot 6 - Training Loss for Different Optimizers"
)

In [ ]:
plot_val_accuracy_comparison(
    optimizer_histories,
    "Plot 7 - Validation Accuracy for Different Optimizers"
)

In [ ]:

learning_rates = [1e-3, 1e-4]

lr_results = []

for lr in learning_rates:

    print("Learning rate:", lr)

    model = build_mobilenet_model(
        initialization="he",
        dropout_rate=0,
        l2_strength=0,
        use_bn=True,
        optimizer_name="adam",
        learning_rate=lr,
        train_base=False
    )

    history, _ = train_model(
        model,
        train_ds,
        val_ds
    )

    lr_results.append({
        "Learning Rate": lr,
        "Validation Accuracy":
            max(history.history["val_accuracy"]) * 100
    })

lr_df = pd.DataFrame(lr_results)

display(lr_df)

In [ ]:

plt.figure(figsize=(8, 5))

plt.plot(
    lr_df["Learning Rate"],
    lr_df["Validation Accuracy"],
    marker="o"
)

plt.xscale("log")

plt.xlabel("Learning Rate")
plt.ylabel("Validation Accuracy (%)")
plt.title("Plot 8 - Learning Rate vs Validation Accuracy")
plt.grid(True)

plt.show()

In [ ]:

batch_sizes = [16, 32, 64]

batch_results = []

for batch_size in batch_sizes:

    print("Batch size:", batch_size)

    current_train_ds = prepare_dataset(
        ds_train,
        batch_size=batch_size,
        shuffle=True
    )

    current_val_ds = prepare_dataset(
        ds_val,
        batch_size=batch_size,
        shuffle=False
    )

    model = build_mobilenet_model(
        initialization="he",
        dropout_rate=0,
        l2_strength=0,
        use_bn=True,
        optimizer_name="adam",
        learning_rate=1e-3,
        train_base=False
    )

    history, _ = train_model(
        model,
        current_train_ds,
        current_val_ds
    )

    batch_results.append({
        "Batch Size": batch_size,
        "Validation Accuracy":
            max(history.history["val_accuracy"]) * 100
    })

batch_df = pd.DataFrame(batch_results)

display(batch_df)

In [ ]:

plt.figure(figsize=(8, 5))

plt.plot(
    batch_df["Batch Size"],
    batch_df["Validation Accuracy"],
    marker="o"
)

plt.xlabel("Batch Size")
plt.ylabel("Validation Accuracy (%)")
plt.title("Plot 9 - Batch Size vs Validation Accuracy")
plt.grid(True)

plt.show()

In [ ]:

dropout_rates = [0, 0.25, 0.5]

dropout_results = []

for dropout in dropout_rates:

    print("Dropout:", dropout)

    model = build_mobilenet_model(
        initialization="he",
        dropout_rate=dropout,
        l2_strength=0,
        use_bn=True,
        optimizer_name="adam",
        learning_rate=1e-3,
        train_base=False
    )

    history, _ = train_model(
        model,
        train_ds,
        val_ds
    )

    dropout_results.append({
        "Dropout Rate": dropout,
        "Validation Accuracy":
            max(history.history["val_accuracy"]) * 100
    })

dropout_df = pd.DataFrame(dropout_results)

display(dropout_df)

In [ ]:

plt.figure(figsize=(8, 5))

plt.plot(
    dropout_df["Dropout Rate"],
    dropout_df["Validation Accuracy"],
    marker="x"
)

plt.xlabel("Dropout Rate")
plt.ylabel("Validation Accuracy (%)")
plt.title("Plot 10 - Dropout Rate vs Validation Accuracy")
plt.grid(True)

plt.show()

In [ ]:

feature_extraction_model = build_mobilenet_model(
    initialization="he",
    dropout_rate=0.25,
    l2_strength=0,
    use_bn=True,
    optimizer_name="adam",
    learning_rate=1e-3,
    train_base=False
)

feature_history, feature_time = train_model(
    feature_extraction_model,
    train_ds,
    val_ds,
    epochs=EPOCHS
)

In [ ]:

fine_tune_model = build_mobilenet_model(
    initialization="he",
    dropout_rate=0.25,
    l2_strength=0,
    use_bn=True,
    optimizer_name="adam",
    learning_rate=1e-5,
    train_base=True,
    fine_tune_layers=20
)

fine_history, fine_time = train_model(
    fine_tune_model,
    train_ds,
    val_ds,
    epochs=EPOCHS
)

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    np.array(feature_history.history["val_accuracy"]) * 100,
    label="Feature Extraction"
)

plt.plot(
    np.array(fine_history.history["val_accuracy"]) * 100,
    label="Fine-Tuning"
)

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy (%)")
plt.title("Plot 11 - Feature Extraction vs Fine-Tuning")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:

plt.figure(figsize=(10, 6))

plt.plot(
    feature_history.history["loss"],
    label="Feature Extraction - Train"
)

plt.plot(
    feature_history.history["val_loss"],
    label="Feature Extraction - Validation"
)

plt.plot(
    fine_history.history["loss"],
    label="Fine-Tuning - Train"
)

plt.plot(
    fine_history.history["val_loss"],
    label="Fine-Tuning - Validation"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Plot 12 - Training and Validation Loss")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:

candidate_configs = {

    "C1_Baseline": {
        "learning_rate": 1e-3,
        "batch_size": 32,
        "dropout": 0,
        "optimizer": "adam"
    },

    "C2_Dropout": {
        "learning_rate": 1e-3,
        "batch_size": 32,
        "dropout": 0.25,
        "optimizer": "adam"
    },

    "C3_LowLR": {
        "learning_rate": 1e-4,
        "batch_size": 32,
        "dropout": 0.25,
        "optimizer": "adam"
    },

    "C4_SGD": {
        "learning_rate": 1e-3,
        "batch_size": 32,
        "dropout": 0.25,
        "optimizer": "sgd"
    }
}

candidate_configs

In [ ]:

def dataset_to_numpy(dataset):

    images = []
    labels = []

    for image, label in dataset:

        image = tf.image.resize(
            image,
            IMG_SIZE
        )

        images.append(image.numpy())
        labels.append(label.numpy())

    images = np.array(images)
    labels = np.array(labels)

    return images, labels


X_cv, y_cv = dataset_to_numpy(ds_train)

print("X:", X_cv.shape)
print("y:", y_cv.shape)

In [ ]:
X_cv = preprocess_input(X_cv.astype(np.float32))

print(
    "Range:",
    X_cv.min(),
    X_cv.max()
)

In [ ]:

def run_5fold_cv(config):

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED
    )

    fold_accuracies = []
    fold_times = []

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X_cv, y_cv),
        start=1
    ):

        print("Fold:", fold)
        X_train_fold = X_cv[train_idx]
        y_train_fold = y_cv[train_idx]

        X_val_fold = X_cv[val_idx]
        y_val_fold = y_cv[val_idx]

        train_dataset = tf.data.Dataset.from_tensor_slices(
            (X_train_fold, y_train_fold)
        ).shuffle(
            1000,
            seed=SEED
        ).batch(
            config["batch_size"]
        ).prefetch(AUTOTUNE)

        val_dataset = tf.data.Dataset.from_tensor_slices(
            (X_val_fold, y_val_fold)
        ).batch(
            config["batch_size"]
        ).prefetch(AUTOTUNE)

        model = build_mobilenet_model(
            initialization="he",
            dropout_rate=config["dropout"],
            l2_strength=0,
            use_bn=True,
            optimizer_name=config["optimizer"],
            learning_rate=config["learning_rate"],
            train_base=False
        )

        start = time.time()

        history = model.fit(
            train_dataset,
            validation_data=val_dataset,
            epochs=EPOCHS,
            verbose=1
        )

        elapsed = time.time() - start

        best_accuracy = max(
            history.history["val_accuracy"]
        ) * 100

        fold_accuracies.append(best_accuracy)
        fold_times.append(elapsed)

        print(
            f"Fold {fold} accuracy: "
            f"{best_accuracy:.2f}%"
        )

    mean_accuracy = np.mean(fold_accuracies)
    std_accuracy = np.std(
        fold_accuracies,
        ddof=1
    )

    mean_time = np.mean(fold_times)

    return {
        "fold_accuracies": fold_accuracies,
        "mean": mean_accuracy,
        "std": std_accuracy,
        "time": mean_time
    }

In [ ]:

cv_results = {}

for name, config in candidate_configs.items():

    print("CONFIGURATION:", name)
    cv_results[name] = run_5fold_cv(config)

In [ ]:

cv_table = []

for name, result in cv_results.items():

    row = {
        "Configuration": name,
        "Fold 1": result["fold_accuracies"][0],
        "Fold 2": result["fold_accuracies"][1],
        "Fold 3": result["fold_accuracies"][2],
        "Fold 4": result["fold_accuracies"][3],
        "Fold 5": result["fold_accuracies"][4],
        "Mean Accuracy": result["mean"],
        "Std": result["std"],
        "Mean Training Time": result["time"]
    }

    cv_table.append(row)

cv_df = pd.DataFrame(cv_table)

display(cv_df)

In [ ]:

plt.figure(figsize=(10, 6))

plt.bar(
    cv_df["Configuration"],
    cv_df["Mean Accuracy"],
    yerr=cv_df["Std"],
    capsize=4
)

plt.ylabel("Mean Validation Accuracy (%)")
plt.xlabel("Configuration")
plt.title("5-Fold Cross-Validation Accuracy")

plt.xticks(rotation=30)

plt.grid(
    axis="y",
    alpha=0.1
)

plt.show()

In [ ]:

best_idx = cv_df["Mean Accuracy"].idxmax()

best_configuration_name = cv_df.loc[
    best_idx,
    "Configuration"
]

best_configuration = candidate_configs[
    best_configuration_name
]

print("Best configuration:")
print(best_configuration_name)

print("\nParameters:")
print(best_configuration)

In [ ]:

final_train_ds = prepare_dataset(
    ds_train_full,
    batch_size=best_configuration["batch_size"],
    shuffle=True
)

final_test_ds = prepare_dataset(
    ds_test,
    batch_size=best_configuration["batch_size"],
    shuffle=False
)

final_model = build_mobilenet_model(
    initialization="he",
    dropout_rate=best_configuration["dropout"],
    l2_strength=0,
    use_bn=True,
    optimizer_name=best_configuration["optimizer"],
    learning_rate=best_configuration["learning_rate"],
    train_base=False
)

final_history, final_training_time = train_model(
    final_model,
    final_train_ds,
    final_test_ds,
    epochs=EPOCHS
)

In [ ]:

final_train_ds = prepare_dataset(
    ds_train_full,
    batch_size=best_configuration["batch_size"],
    shuffle=True
)

# Use the existing validation data for monitoring
final_val_ds = prepare_dataset(
    ds_val,
    batch_size=best_configuration["batch_size"],
    shuffle=False
)

final_test_ds = prepare_dataset(
    ds_test,
    batch_size=best_configuration["batch_size"],
    shuffle=False
)

final_model = build_mobilenet_model(
    initialization="he",
    dropout_rate=best_configuration["dropout"],
    l2_strength=0,
    use_bn=True,
    optimizer_name=best_configuration["optimizer"],
    learning_rate=best_configuration["learning_rate"],
    train_base=False
)

final_history, final_training_time = train_model(
    final_model,
    final_train_ds,
    final_val_ds,
    epochs=EPOCHS
)

In [ ]:

y_true = []
y_pred = []

for images, labels in final_test_ds:

    predictions = final_model.predict(
        images,
        verbose=0
    )

    predicted_labels = np.argmax(
        predictions,
        axis=1
    )

    y_true.extend(labels.numpy())
    y_pred.extend(predicted_labels)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

In [ ]:

test_accuracy = accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

num_parameters = final_model.count_params()

final_results = pd.DataFrame({
    "Metric": [
        "Mean CV Accuracy",
        "CV Standard Deviation",
        "Test Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "Training Time (seconds)",
        "Number of Parameters"
    ],

    "Value": [
        cv_df.loc[
            best_idx,
            "Mean Accuracy"
        ],

        cv_df.loc[
            best_idx,
            "Std"
        ],

        test_accuracy * 100,

        precision * 100,

        recall * 100,

        f1 * 100,

        final_training_time,

        num_parameters
    ]
})

display(final_results)

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred
)

class_names = [
    ds_info.features["label"].int2str(i)
    for i in range(NUM_CLASSES)
]

plt.figure(figsize=(18, 16))

sns.heatmap(
    cm,
    annot=True,          # Show values
    fmt="d",             # Display as integers
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    linewidths=0.5
)

plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.title("Plot 14 - Confusion Matrix")

plt.xticks(
    rotation=90,
    fontsize=8
)

plt.yticks(
    rotation=0,
    fontsize=8
)

plt.tight_layout()
plt.show()

In [ ]:

report = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

classification_df = pd.DataFrame(report).T

display(classification_df)

In [ ]:

misclassified_indices = np.where(
    y_true != y_pred
)[0]

print(
    "Number of misclassified images:",
    len(misclassified_indices)
)

In [ ]:

original_test_images = []

original_test_labels = []

for image, label in ds_test:

    image = tf.image.resize(
        image,
        IMG_SIZE
    )

    original_test_images.append(
        image.numpy().astype(np.uint8)
    )

    original_test_labels.append(
        label.numpy()
    )

original_test_images = np.array(
    original_test_images
)

original_test_labels = np.array(
    original_test_labels
)

In [ ]:

num_images = min(
    12,
    len(misclassified_indices)
)

plt.figure(figsize=(14, 10))

for i in range(num_images):

    idx = misclassified_indices[i]

    plt.subplot(3, 4, i + 1)

    plt.imshow(
        original_test_images[idx]
    )

    true_name = class_names[
        y_true[idx]
    ]

    pred_name = class_names[
        y_pred[idx]
    ]

    plt.title(
        f"True: {true_name}\nPred: {pred_name}",
        fontsize=9
    )

    plt.axis("off")

plt.suptitle(
    "Plot 15 - Representative Misclassified Images"
)

plt.tight_layout()
plt.show()

In [ ]:

cm_without_diagonal = cm.copy()

np.fill_diagonal(
    cm_without_diagonal,
    0
)

top_confusions = []

for _ in range(10):

    idx = np.unravel_index(
        np.argmax(cm_without_diagonal),
        cm_without_diagonal.shape
    )

    true_class = class_names[idx[0]]
    predicted_class = class_names[idx[1]]
    count = cm_without_diagonal[idx]

    top_confusions.append({
        "True Class": true_class,
        "Predicted Class": predicted_class,
        "Count": count
    })

    cm_without_diagonal[idx] = 0

top_confusions_df = pd.DataFrame(
    top_confusions
)

display(top_confusions_df)

In [ ]:


overall_results = []

# ------------------------------------------------------------
# Baseline
# ------------------------------------------------------------

overall_results.append({
    "Configuration": "Baseline",
    "CV Accuracy": np.nan,
    "CV SD": np.nan,
    "Test Accuracy": np.nan,
    "Training Time": baseline_time
})

# ------------------------------------------------------------
# Best Initialization
# ------------------------------------------------------------

best_init = init_results_df[
    "Best Validation Accuracy"
].idxmax()

overall_results.append({
    "Configuration": "Best Initialization",
    "CV Accuracy": np.nan,
    "CV SD": np.nan,
    "Test Accuracy": np.nan,
    "Training Time":
        init_results_df.loc[
            best_init,
            "Training Time"
        ]
})

# ------------------------------------------------------------
# Best Regularization
# ------------------------------------------------------------

best_reg = regularization_df[
    "Best Val Accuracy"
].idxmax()

overall_results.append({
    "Configuration": "Best Regularization",
    "CV Accuracy": np.nan,
    "CV SD": np.nan,
    "Test Accuracy": np.nan,
    "Training Time":
        regularization_df.loc[
            best_reg,
            "Training Time"
        ]
})

# ------------------------------------------------------------
# Best Optimizer
# ------------------------------------------------------------

best_optimizer = optimizer_df[
    "Best Val Accuracy"
].idxmax()

overall_results.append({
    "Configuration": "Best Optimizer",
    "CV Accuracy": np.nan,
    "CV SD": np.nan,
    "Test Accuracy": np.nan,
    "Training Time":
        optimizer_df.loc[
            best_optimizer,
            "Training Time"
        ]
})

# ------------------------------------------------------------
# Best Hyperparameters
# ------------------------------------------------------------

overall_results.append({
    "Configuration": "Best Hyperparameters",
    "CV Accuracy":
        cv_df.loc[
            best_idx,
            "Mean Accuracy"
        ],

    "CV SD":
        cv_df.loc[
            best_idx,
            "Std"
        ],

    "Test Accuracy":
        test_accuracy * 100,

    "Training Time":
        final_training_time
})

# ------------------------------------------------------------
# Fine Tuned
# ------------------------------------------------------------

overall_results.append({
    "Configuration": "Fine-Tuned Model",
    "CV Accuracy": np.nan,
    "CV SD": np.nan,
    "Test Accuracy": np.nan,
    "Training Time": fine_time
})

overall_df = pd.DataFrame(
    overall_results
)

display(overall_df)

In [ ]:

hyperparameter_summary = pd.concat(
    [
        lr_df.rename(
            columns={
                "Learning Rate": "Value",
                "Validation Accuracy":
                    "Validation Accuracy"
            }
        ).assign(
            Hyperparameter="Learning Rate"
        )[[
            "Hyperparameter",
            "Value",
            "Validation Accuracy"
        ]],

        batch_df.rename(
            columns={
                "Batch Size": "Value"
            }
        ).assign(
            Hyperparameter="Batch Size"
        )[[
            "Hyperparameter",
            "Value",
            "Validation Accuracy"
        ]],

        dropout_df.rename(
            columns={
                "Dropout Rate": "Value"
            }
        ).assign(
            Hyperparameter="Dropout Rate"
        )[[
            "Hyperparameter",
            "Value",
            "Validation Accuracy"
        ]]
    ],
    ignore_index=True
)

display(hyperparameter_summary)

In [ ]:
# ============================================================
# CELL 36 - SAVE RESULTS
# ============================================================

init_results_df.to_csv(
    "initialization_results.csv"
)

regularization_df.to_csv(
    "regularization_results.csv"
)

optimizer_df.to_csv(
    "optimizer_results.csv"
)

lr_df.to_csv(
    "learning_rate_results.csv"
)

batch_df.to_csv(
    "batch_size_results.csv"
)

dropout_df.to_csv(
    "dropout_results.csv"
)

cv_df.to_csv(
    "cross_validation_results.csv"
)

final_results.to_csv(
    "final_model_results.csv"
)

overall_df.to_csv(
    "overall_results.csv"
)

print("All result files saved.")

In [ ]:

print("=" * 70)
print("FINAL MODEL SUMMARY")
print("=" * 70)

print(
    f"Best Configuration : {best_configuration_name}"
)

print(
    f"CV Accuracy        : "
    f"{cv_df.loc[best_idx, 'Mean Accuracy']:.2f}%"
)

print(
    f"CV Standard Dev.   : "
    f"{cv_df.loc[best_idx, 'Std']:.2f}%"
)

print(
    f"Test Accuracy      : "
    f"{test_accuracy * 100:.2f}%"
)

print(
    f"Precision          : "
    f"{precision * 100:.2f}%"
)

print(
    f"Recall             : "
    f"{recall * 100:.2f}%"
)

print(
    f"F1 Score           : "
    f"{f1 * 100:.2f}%"
)

print(
    f"Training Time      : "
    f"{final_training_time:.2f} seconds"
)

print(
    f"Parameters         : "
    f"{num_parameters:,}"
)

print("=" * 70)

In [ ]:
# ============================================================
# ADDITIONAL CELL - Section 13 Overall Results (no NaNs)
# Place this after the existing overall_results cell
# ============================================================

# Best init name + metrics
best_init_name = init_results_df["Best Validation Accuracy"].idxmax()
best_init_val  = init_results_df.loc[best_init_name, "Best Validation Accuracy"]
best_init_time = init_results_df.loc[best_init_name, "Training Time"]

# Best regularization
best_reg_name = regularization_df["Best Val Accuracy"].idxmax()
best_reg_val  = regularization_df.loc[best_reg_name, "Best Val Accuracy"]
best_reg_time = regularization_df.loc[best_reg_name, "Training Time"]

# Best optimizer
best_opt_name = optimizer_df["Best Val Accuracy"].idxmax()
best_opt_val  = optimizer_df.loc[best_opt_name, "Best Val Accuracy"]
best_opt_time = optimizer_df.loc[best_opt_name, "Training Time"]

# Fine-tuning best val acc
fine_best_val = max(fine_history.history["val_accuracy"]) * 100

# Baseline best val acc (from its history)
baseline_best_val = max(baseline_history.history["val_accuracy"]) * 100

# Selected CV config
best_cv_mean = cv_df.loc[best_idx, "Mean Accuracy"]
best_cv_std  = cv_df.loc[best_idx, "Std"]
best_cv_test = test_accuracy * 100
best_cv_time = final_training_time

overall_clean = [
    {
        "Configuration": "Baseline",
        "CV Accuracy": "-",
        "SD": "-",
        "Test / Val Accuracy": f"{baseline_best_val:.2f}",
        "Training Time (s)": f"{baseline_time:.2f}"
    },
    {
        "Configuration": f"Best Initialization ({best_init_name})",
        "CV Accuracy": "-",
        "SD": "-",
        "Test / Val Accuracy": f"{best_init_val:.2f}",
        "Training Time (s)": f"{best_init_time:.2f}"
    },
    {
        "Configuration": f"Best Regularization ({best_reg_name})",
        "CV Accuracy": "-",
        "SD": "-",
        "Test / Val Accuracy": f"{best_reg_val:.2f}",
        "Training Time (s)": f"{best_reg_time:.2f}"
    },
    {
        "Configuration": f"Best Optimizer ({best_opt_name})",
        "CV Accuracy": "-",
        "SD": "-",
        "Test / Val Accuracy": f"{best_opt_val:.2f}",
        "Training Time (s)": f"{best_opt_time:.2f}"
    },
    {
        "Configuration": "Best Hyperparameters (C1_Baseline)",
        "CV Accuracy": f"{best_cv_mean:.2f}",
        "SD": f"{best_cv_std:.2f}",
        "Test / Val Accuracy": f"{best_cv_test:.2f}",
        "Training Time (s)": f"{best_cv_time:.2f}"
    },
    {
        "Configuration": "Fine-Tuned Model",
        "CV Accuracy": "-",
        "SD": "-",
        "Test / Val Accuracy": f"{fine_best_val:.2f}",
        "Training Time (s)": f"{fine_time:.2f}"
    },
]

overall_df_clean = pd.DataFrame(overall_clean)
display(overall_df_clean)

# ---- LaTeX-ready table string ----
print("\n% ---- Copy this into the report (Section 13) ----")
print(r"\begin{tabular}{|l|c|c|c|c|}")
print(r"\hline")
print(r"Configuration & CV Accuracy & SD & Test / Val Accuracy & Training Time (s) \\")
print(r"\hline")
for _, row in overall_df_clean.iterrows():
    print(f"{row['Configuration']} & {row['CV Accuracy']} & {row['SD']} & {row['Test / Val Accuracy']} & {row['Training Time (s)']} \\\\")
print(r"\hline")
print(r"\end{tabular}")